In [ ]:
import os

import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import (
        log_metric,
        log_params,
        set_experiment,
        start_run,
)
from mlflow.models.signature import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest as IsoF
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from skopt import BayesSearchCV
from skopt.space import Categorical

import dataprocessing as dc


In [ ]:
df = pd.read_csv("data/email_data.csv")
newdata = dc.Data_class(data = df, )
df = newdata.data
df["Category"] = np.where(df['Category'] == 'spam', -1, 1)


In [ ]:
y = df["Category"]
X = df.drop(
    ["Category", "Message", "word_count"],
    axis = 1
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify= y,
    random_state = 42
)


In [ ]:
numeric_cols = X.select_dtypes(
    include = ["number"]
).columns
numeric_pipe = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),
        ("encoder",MinMaxScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_cols)
    ],
    remainder='drop'  # or 'passthrough' if you want to keep unlisted columns
)

In [ ]:
model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("model", IsoF(random_state = 42))
    ]
)

In [ ]:


#determining number of features max
n_features = X_train.shape[1]

#determing rate of spam within data
spam_rate = float((y_train == -1).mean())

search_space = {
    "model__n_estimators": Categorical([100, 150, 200, 300, 400]),
    "model__max_samples": Categorical(['auto', 64, 128, 256, 512]),
    "model__max_features": Categorical([
        1.0,
        0.5,
        0.75,
        int(n_features ** 0.5),
        int(np.log2(n_features)),
    ]),
    "model__contamination": Categorical([
        "auto",
        spam_rate,
        spam_rate * 0.5,
        spam_rate * 1.5,
        0.1,
        0.2,
        0.3,
    ]),
}

In [ ]:
spam_scorer = make_scorer(f1_score, pos_label=-1)
set_experiment("IsolationForest")
#%%capture #uncomment to suppress output
with start_run(
    run_name="IsolationForest_BayesianSearch_AllConfigs",
    log_system_metrics=True,
):

        bayes_search_cv = BayesSearchCV(
            estimator=model,
            search_spaces=search_space,
            n_iter=32,
            cv=5,
            scoring=spam_scorer,
            random_state=42,
            verbose=0,
        )

        # Fit the Bayesian search
        bayes_search_cv.fit(X_train, y_train)

        # Best estimator from the search
        best_model = bayes_search_cv.best_estimator_

        # IsolationForest outputs: -1 = anomaly, 1 = inlier.
        # Remap so the anomaly (spam) is the positive class for metric computation.
        raw_pred = best_model.predict(X_test)
        y_pred = (raw_pred == -1).astype(int)

        # decision_function is HIGHER for normal points, so negate:
        # higher score -> more anomalous -> correct orientation for AUC / PR-AUC
        y_score = -best_model.decision_function(X_test)

        # Metrics
        acc       = accuracy_score(y_test, y_pred)
        pr_auc    = average_precision_score(y_test, y_score)
        roc_auc   = roc_auc_score(y_test, y_score)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)

        # Log parameters & metrics
        log_params(bayes_search_cv.best_params_)
        log_metric("accuracy", acc)
        log_metric("Average Precision", pr_auc)
        log_metric("roc_auc", roc_auc)
        log_metric("precision", precision)
        log_metric("recall", recall)
        log_metric("f1", f1)
        log_metric("best_cv_score", bayes_search_cv.best_score_)

        # Optional but best practice: define input example and signature
        input_example = pd.DataFrame(X_test[:1], columns=X_test.columns)
        signature = infer_signature(X_test, y_score)   # scores, not labels

        # Log candidate model in MLflow experiment
        mlflow.sklearn.log_model(
            best_model, name="candidate_model",
            input_example=input_example,
            signature=signature,
            serialization_format="cloudpickle",
        )

In [ ]:
print("Best Parameters:", bayes_search_cv.best_params_)
print("Best Score:", bayes_search_cv.best_score_)

# Predict with best model
y_pred = bayes_search_cv.best_estimator_.predict(X_test)